WORD EMBEDDINGS IN TENSORFLOW E KERAS: RAPPRESENTAZIONI DENSE E DINAMICHE

Con TensorFlow/Keras gli embedding non devono necessariamente essere vettori già pronti come GloVo o Word2Vec: puoi inserire un layer embededing nella rete e lasciare che sia il modello a imparare i vettori durante il training.
Prendiamo la frase:
"il film è bello"
dopo la tokenizzazione
il -> 14
file -> 58
è -> 9
bello -> 125
quindi il modello riceve: [14,58,9,125]
Questi numeri non rappresentano il significato, sono soltanto ID del vocabolario
Qui entra il layer embedding
keras.layers.embedding(...)
qui ogni token viene rappresentato con un vettore di n numeri (esempio di 128 numeri)
quindi:
token ID 58 -> embedding -> [0.21,-0.37,...] (128 valori)
Se hai una frase di 5 token e ogni embedding ha 128 valori, l'output dell'embedding layer sarà una matrice 5 token x 128 dimensioni

Questa si chiama rappresentazione densa
Perchè densa?
perchè rispetto a Bag of Words o TF-IDF non hai un vettore enorme pieno di zeri.

La parte ancora più importante è il termine dinamiche
Quando crei gli embedding i valori di embedding all'inizio vengono inizializzati dal layer, durante il training la backpropagation modifica questi valori, e dopo molti esempi il vettore contiene una rappresentazione semantica della parola. Quindi il modello impare gli embedding insieme al resto della rete.
La LSTM o la GRU lavorano direttamente su questi vettori e non sugli ID
Questi embedding sono specifici per il testo che si sta analizzando e per il task 
Esempio se addestro per sentiment analysis gli embedding vengono ottimizzati per aiutare a distinguere positivo/negativo.
Se addestrassi la stessa architettura per ham/spam otteresti embedding differenti, anche usando lo stesso vocabolario.
Questo li distingue dagli embedding generici preaddestrati
Con GloVe ho un grande corpus che ha generato embedding generici
Con Keras gli embedding sono calcolati sul mio training e l'embedding è specifico. 
Ma puoi anche cambiare le cose, esempio puoi prendere embedding preaddestrati, per esempio Glove, caricarli nella matrice dell'embedding layer e decidere se lasciarli fissi oppure continuare l'addestramento (trainable=True).

Quando diciamo che gli embedding Keras sono 'dinamici' perchè cambiano durante il training, non significa necessariamente 'contextual embeddings' come BERT.
Sono due concetti differenti
Con un normale layer kersa (kersa.layer.Embedding(...))
dopo il training: film ha lo stesso vettore base ogni volta che compare la parola film anche se di significato diverso (es. "un bel film", "pellicola per il fils fotografico"). In questo caso si dice che è un embedding statico rispetto al contesto.

BERT invece produce due rappresentazioni contestuali, stessa parola ma con significato diverso in base al contesto, produce una rappresentazione diversa.

Il layer Embedding di TensorFlow/Keras trasforma gli ID dei token in vettori densi di dimensione fissa. I valori della matrice di embedding sono parametri del modello e possono essere appresi durante il training tramite backpropagation, oppure inizializzati con vettori preaddestrati e successivamente congelati o ulteriormente addestrati.

Gli embedding sono un concetto che ha cambiato per sempre il modo in cui le macchina apprendono il linguaggio umano.

Per costruire un layer di embedding dobbiamo defnire alcuni parametri
- Input_dim: rappresenta la dimensione del vocabolario
- Output_dim: è la dimensione del vettore in cui le parole vengono rappresentate
- Input_lenght: è la lunghezza fissa delle sequenze in ingresso (necessaria per i layer Dense successivi)
- Trainable: parametro boolenao che decide se aggiornare i vettori durante la backpropagation

Quando passiamo un dato al layer, all'inizio ogni parola è un vettore casuale. Non c'è alcuna relazione semantica tra 'gatto' e 'cane'. Per lui, cane e gatto, all'inizio, sono posizionati a caso nello spazio.
Mentre il modello legge migliaia di frasi, nota che certe parole appaiono in contesti simili e ne avvicina i vettori (apprendimento dinamico). Il vantaggio non è solo concettuale ma anche computazionale.
L'Embedding layer evita la moltiplicazione tra matrici giganti tipica del One-Hot encoding, usando un'indicizzazione diretta molto efficiente.

Ogni parola diventa un punto in uno spazio di d dimensioni. L'output èer una sequenza di indici sarà una matrice di forma, che i layer successivi potranno elaborare per capire un sentimento o altro.

Tuttavia le frasi non sono tutte lunghe uguali, entrano in gioco Padding e Masking
Le reti neurali si aspettano frasi delle stesse dimensioni, ed è qui che entrano in gioco le tecniche di uniformazione.
- pad_sequences: aggiunge degli zeri (padding) per allungare le frasi corte e taglia quelle troppo lunghe
Ma non vogliamo che la reti pensi che queli zero siano importanti, usanto mask_zero=True nel layer embedding è come dire alla rete di ignorare questa maschera.

Senza una maschera una rete ricorrente proverebbe a ricordare anche gli zeri degradando il segnale.
Keras gestire internamente un boolean mask tensor che 'spegne' i calcoli dove sono presenti i pad.
Spesso usiamo il pre-padding perchè l'informazione più fresca è quella che conta di più

In termni formali il passing trasforma un vettore di lunghezza variabile in un tensore di lunghezza fissa. l'operazione assicura che il batch abbia dimensioni coerenti per il calcolo parallelu su GPU 

Ma dobbiamo davvero imparare da zero tutto ogni volta
e qui entra in gioco il transfer learning.

Spesso non abbiamo abbastanza dati per far imparare le relazioni semantiche da zero. Possiamo 'iniettare' conoscenza esterna.
Si possono mappare matrice di pesi esterne (come Glove) all'interno del nostro layer di Embedding in Keras. Dando alla rete una cultura generale istantanea.

Implementazione Pratica
- Embedding Matrix: creiamo una matrice numpy che faccia da ponte. Per ogni parola del nostro tokenizer cerchiamo il corrisondente nel file pre-addestrato. Se la parola esiste copiamo il vettore, e se non esite inziamo con un vettore casuale.
Infine carichiamo questa matrice nel layer embedding utilizzanod embeddings_initializer.
E' come installare  un database di conoscenza già pronto. Una volta caricati i pesi dobbiamo decidere la nostra strategia.
- Se abbiamo pochi dati useriemo gli static-embeddings e congeliamo i pesi. La rete userà la conoscenza già appresa ed i pesi rimangono fissi.
- Non-static Embeddings: i pesi pre-addestrati sono il punto di partenza, ma vengono rifiniti. Ottimo per adattarsi a gerchi specifici. Questo se abbiamo un dataset discreto e proecediamo con il fine-tunning. I pesi pre-addestrati sono solo il punto di partenza e la rete li modificherà leggermente per adattarli al nostro contesto specifico.
- Out-of-Vocabulary: le parole non presenti nel modello pre-addestrato vengono solitamente inizializzate con vettori di zeri o valori casuali.

Vantaggi del Transfer Learning
Accellera la convergenza, e permette al modello di generalizzare, soprattutto su parole che non ha mai visto durante il training locale.
Il modello sa già che 'ottimo' è simile a 'eccellente' anche se  non ha mai visto 'eccellente' nel nostor piccolo dataset di addestramento.

In [1]:
"""
========================================================================================
WORD EMBEDDINGS E TRANSFER LEARNING REALE (KERAS 3 + PYTORCH)
========================================================================================
Obiettivo: Trasformare parole umane in vettori dotati di significato usando pesi GloVe reali.

INTERAZIONI CHIAVE:
1. TextVectorization: Il "Portinaio" che trasforma il testo grezzo in numeri (indici).
2. Gensim: Il "Fornitore" che scarica i vettori pre-addestrati da miliardi di documenti.
3. Embedding Matrix: Il "Ponte" che associa i nostri indici locali ai vettori di Gensim.
4. Embedding Layer: Il "Cuore" della rete che memorizza e proietta i vettori nello spazio.
========================================================================================
"""

import os

# FASE 0: SETUP DEL MOTORE DI CALCOLO
# Keras 3 è agnostico: qui scegliamo PyTorch come motore per le operazioni tensoriali.
os.environ["KERAS_BACKEND"] = "torch"

import keras
from keras import layers
import numpy as np

# Verifichiamo la presenza di Gensim (essenziale per scaricare i vettori GloVe reali)
try:
    import gensim.downloader as api
except ImportError:
    print("[ERRORE] Libreria 'gensim' mancante. Esegui: pip install gensim")
    exit()

def prepare_data():
    """
    FASE 1: PREPARAZIONE DEL TESTO E DEL VOCABOLARIO (Slide 2, 7, 8)
    ---------------------------------------------------------------
    Qui trasformiamo frasi di lunghezza diversa in una matrice numerica fissa.
    """
    # Piccoli esempi di sentiment analysis (Positivo vs Negativo)
    testi = [
        "The movie was absolutely fantastic and worth watching",
        "Terrible service and very bad food quality",
        "An incredible experience that I loved",
        "I did not like it at all, very boring"
    ]
    # Labels binarie: 1=Felice, 0=Triste
    labels = np.array([1, 0, 1, 0], dtype="float32")
    
    # TextVectorization (Il nostro sarto professionista):
    # - max_tokens: Considera solo le 1000 parole più frequenti.
    # - output_mode='int': Ogni parola diventa un intero unico.
    # - output_sequence_length=10: Fa PADDING (aggiunge zeri) o CLIP (taglia) a 10 parole.
    vectorizer = layers.TextVectorization(
        max_tokens=1000,
        output_mode="int",
        output_sequence_length=10, 
    )
    
    # 'adapt' legge i testi e costruisce internamente il vocabolario (mappa parola <-> indice)
    vectorizer.adapt(testi)
    
    return testi, labels, vectorizer

def build_real_transfer_model(vectorizer, embedding_dim=50):
    """
    FASE 2: COSTRUZIONE DEL MODELLO E TRASFUSIONE DI CONOSCENZA (Slide 11, 12, 13)
    -------------------------------------------------------------------------
    Qui carichiamo GloVe e iniettiamo i suoi pesi nel nostro layer Keras.
    """
    # Recuperiamo il vocabolario creato dal vectorizer
    vocab = vectorizer.get_vocabulary()
    num_tokens = len(vocab)
    
    # --- PASSO A: Caricamento Pesi GloVe (Conoscenza Esterna) ---
    print("\n[INFO] Download di 'glove-wiki-gigaword-50' (Conoscenza distillata da miliardi di parole)...")
    glove_vectors = api.load("glove-wiki-gigaword-50")
    
    # --- PASSO B: Creazione Matrice Ponte (La Trasfusione) ---
    # Creiamo una tabella vuota (zeri) di forma (NumeroParole x 50 Dimensioni)
    embedding_matrix = np.zeros((num_tokens, embedding_dim))
    
    hits, misses = 0, 0
    for i, word in enumerate(vocab):
        # Cerchiamo se la parola del nostro piccolo vocabolario esiste in GloVe
        if glove_vectors.has_index_for(word):
            # Se esiste, copiamo il vettore densi (il significato semantico)
            embedding_matrix[i] = glove_vectors[word]
            hits += 1
        else:
            # Se non esiste (es. nomi propri strani), la parola rimane con vettore zero
            misses += 1
            
    print(f"[INFO] Trasfusione: {hits} parole mappate, {misses} inizializzate a zero.")

    # --- PASSO C: Architettura del Modello (API Funzionale) ---
    # 1. Ingresso: sequenze di 10 numeri interi
    inputs = layers.Input(shape=(10,), dtype="int32", name="Ingresso_Indici")
    
    # 2. Il Layer Embedding (Il Traduttore Universale)
    # - input_dim: quante parole conosciamo.
    # - output_dim: quante coordinate ha ogni parola (50).
    # - embeddings_initializer: INIETTIAMO qui la nostra matrice ponte di GloVe.
    # - mask_zero=True: Dice alla rete di ignorare i '0' del padding (Slide 9).
    # - trainable=False: CONGELIAMO i pesi. Non vogliamo cambiare la saggezza di GloVe.
    embedding_layer = layers.Embedding(
        input_dim=num_tokens,
        output_dim=embedding_dim,
        embeddings_initializer=keras.initializers.Constant(embedding_matrix),
        mask_zero=True,
        trainable=False, 
        name="Memoria_GloVe"
    )
    
    # Colleghiamo i pezzi:
    x = embedding_layer(inputs)           # (Batch, 10) -> (Batch, 10, 50)
    x = layers.GlobalAveragePooling1D()(x) # (Batch, 10, 50) -> (Batch, 50) - Media della frase
    outputs = layers.Dense(1, activation="sigmoid")(x) # Output finale: probabilità 0-1
    
    # Compilazione
    model = keras.Model(inputs, outputs)
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

# ========================================================================================
# LOGICA DI ESECUZIONE (IL MAIN)
# ========================================================================================

# 1. Prepariamo i dati e il vettorizzatore
testi, labels, vectorizer = prepare_data()

# 2. Creiamo il cervello (Modello) caricando GloVe
# Notate come il modello 'incapsula' la conoscenza pre-addestrata
model = build_real_transfer_model(vectorizer)

# 3. Visione d'insieme del Modello
model.summary()

# --- ISPEZIONE FISICA (L'Anima della Parola) ---
# Vogliamo vedere il vettore numerico della parola 'fantastic'
print("\n--- ISPEZIONE LIVE (SLIDE 6) ---")

# Step 1: Trasformiamo 'fantastic' nel suo indice numerico
# Usiamo keras.ops per gestire correttamente la memoria se siamo su GPU
parola_test = ["fantastic"]
indices_tensore = vectorizer(parola_test)
indices_np = keras.ops.convert_to_numpy(indices_tensore)
idx = int(indices_np[0][0])

# Step 2: Estraiamo il vettore (i pesi) dal Layer Embedding per quell'indice
pesi_embedding = model.get_layer("Memoria_GloVe").get_weights()[0]
vettore = pesi_embedding[idx]

print(f"Parola: '{parola_test[0]}' -> Convertita in Indice: {idx}")
print(f"Prime 5 dimensioni del suo vettore GloVe reale:\n{vettore[:5]}...")

# 4. TEST DI INFERENZA FINALE
# Vediamo se il modello capisce il sentiment di una frase mai vista

# addestrimao il nostro modello
X_train = vectorizer(testi)
y_train = labels

print("\n[INFO] Inizio addestramento...")
model.fit(X_train, y_train, epochs=20, verbose=1)

# eseguiamo l'inferenza
nuova_frase = ["this movie is fantastic"]
test_input = vectorizer(nuova_frase)
predizione = model.predict(test_input, verbose=0)

# Estraiamo il valore scalare dalla predizione
probabilita = float(keras.ops.convert_to_numpy(predizione)[0][0])
sentiment = "POSITIVO" if probabilita > 0.5 else "NEGATIVO"

print(f"\n[RISULTATO]")
print(f"Frase: '{nuova_frase[0]}'")
print(f"Probabilità Positività: {probabilita:.4f} -> Sentiment: {sentiment}")


[INFO] Download di 'glove-wiki-gigaword-50' (Conoscenza distillata da miliardi di parole)...
[==================================================] 100.0% 66.0/66.0MB downloaded
[INFO] Trasfusione: 27 parole mappate, 2 inizializzate a zero.


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Ingresso_Indici     │ (None, 10)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Memoria_GloVe       │ (None, 10, 50)    │      1,450 │ Ingresso_Indici[… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 10)        │          0 │ Ingresso_Indici[… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 50)        │          0 │ Memoria_GloVe[0]… │
│ (GlobalAveragePool… │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 1)         │         51 │ global_average_p… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,501 (5.86 KB)

 Trainable params: 51 (204.00 B)

 Non-trainable params: 1,450 (5.66 KB)


--- ISPEZIONE LIVE (SLIDE 6) ---
Parola: 'fantastic' -> Convertita in Indice: 20
Prime 5 dimensioni del suo vettore GloVe reale:
[ 0.3333    0.30612  -0.63572   0.051507  0.78602 ]...

[INFO] Inizio addestramento...
Epoch 1/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 375ms/step - accuracy: 0.5000 - loss: 0.8005
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.5000 - loss: 0.7967
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.5000 - loss: 0.7930
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.5000 - loss: 0.7893
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5000 - loss: 0.7857
Epoch 6/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5000 - loss: 0.7822
Epoch 7/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5000 - loss: 0.7786
Epoch 8/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5000 - loss: 0.7752
Epoch 9/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5000 - loss: 0.7718
Epoch 10/20
1/1 ━━━━━━━━━━━━━━━━━━━━